# Experiment 1: Residual Networks

## Constructing Tensorflow Datasets

We can load the cropped and pre-split images under `processed_data` into three [`tf.data.Dataset`](https://www.tensorflow.org/api_docs/python/tf/data/Dataset) instances. This is the data structure we'll need to use to contain the normalised image tensors. We must group samples into batches, and enable prefetching (paralellises CPU data loading and GPU model training).

In [5]:
import os
import tensorflow as tf

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# NOTE: Configure as needed to use CUDA / Metal, etc. I believe TensorFlow should detect CUDA out of the box. 
# I have deliberately excluded tensorflow-metal from requirements.txt, install in your venv if using Apple.
# I'm using tensorflow version 2.18.1 on mac.
print(tf.config.list_physical_devices("GPU"))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [6]:
CLASS_NAMES = ["NORMAL", "PNEUMONIA"]
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

def normalise(image, label):
    return tf.cast(image, tf.float32) / 255.0, label

def load_split(path, shuffle=False):
    # Load datasets as tf.data.Dataset from preprocessed image directories
    return tf.keras.utils.image_dataset_from_directory(
        path,
        class_names=CLASS_NAMES,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        color_mode="grayscale",
        shuffle=shuffle,
        seed=42 if shuffle else None
    )

# Normalise pixel values and enable parallel prefetching. We'll shuffle the train set once more.
train_set = (load_split("processed_data/train", shuffle=True)
    .map(normalise, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE))

valid_set = (load_split("processed_data/val")
    .map(normalise, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE))

test_set = (load_split("processed_data/test")
    .map(normalise, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE))

Found 4710 files belonging to 2 classes.
Found 524 files belonging to 2 classes.
Found 624 files belonging to 2 classes.


In [7]:
for images, labels in train_set.take(1): # Quick inspection of a single batch
    print(f"Image batch shape: {images.shape}")  # 32 Samples, 244x244, 1 channel
    print(f"Label batch shape: {labels.shape}")  # 32 labels
    print(f"Pixel range: {images.numpy().min()} - {images.numpy().max()}")
    print(f"Batch labels: {labels.numpy()}")

Image batch shape: (32, 224, 224, 1)
Label batch shape: (32,)
Pixel range: 0.0 - 1.0
Batch labels: [0 1 1 1 1 0 0 1 1 1 1 0 1 1 1 0 0 1 0 0 1 0 1 0 1 1 0 1 0 1 1 1]


2026-05-10 20:12:50.971848: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## Model Construction and Training Loop

Here we construct the 34 layer ResNet. Please see train.py for implementation details. **TL;DR**: residual connections give the network paths that allow gradients to bypass weight layers during backprop. This significantly mitigates the vanishing gradient problem and so allows for construction of much deeper networks. Residual units (essentially just pairs of convolutional layers with a skip connection) are stacked just after an initial 7x7 convolutional and max pooling layer. A global average pooling layer reduces the final 512 feature maps to a vector of 512 dims, which is passed through a single dense unit (using Sigmoid to bound the output between 0 and 1, exactly what we need for binary classification). As such, we also train using a binary cross entropy loss.

First, we can prepare some callbacks to enable model checkpointing and early stopping. Then we can construct, compile and train. Training loops are abstracted in TensorFlow so it's just single `compile` and `fit` calls.

In [12]:
# Create callbacks
CHECKPOINT_DIR = "checkpoints/resnet34"
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    CHECKPOINT_DIR + "/resnet34_epoch-{epoch:02d}.weights.h5",
    save_weights_only=True,
    save_best_only=False, # Save weights for every epoch
    monitor="val_loss",
    mode="min",
    verbose=1
)
    
early_stopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
)

# I'll add a TensorBoard callback later...

In [22]:
import glob
import numpy as np
from models import ResNet34

# Model initialisation and training parameters
resnet34 = ResNet34()
LEARNING_RATE = 1e-4
EPOCHS = 10

# Compile
resnet34.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE # Fixed learning rate
    ),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=["accuracy"]
)

# Load from checkpoints if exists
checkpoints = sorted(glob.glob(f"{CHECKPOINT_DIR}/*.weights.h5"))

if checkpoints:
    latest = checkpoints[-1]
    resnet34(np.zeros((1, 224, 224, 1), dtype="float32")) # Force build of all sublayers
    resnet34.load_weights(latest)
    # Epoch number is contained within the filename
    initial_epoch = int(latest.split("epoch-")[1].split(".")[0])
    print(f"Resuming from {latest}")
else:
    resume_epoch = 0
    print("No checkpoints found, training from scratch")

# Train
history = resnet34.fit(
    train_set,
    validation_data=valid_set,
    epochs=EPOCHS,
    initial_epoch=initial_epoch,
    callbacks=[checkpoint_callback], # Feel free to experiment and add the early stopping callback too
    verbose=1
)

Resuming from checkpoints/resnet34/resnet34_epoch-01.weights.h5
Epoch 2/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 0s 448ms/step - accuracy: 0.7755 - loss: 1.2179
Epoch 2: saving model to checkpoints/resnet34/resnet34_epoch-02.weights.h5
148/148 ━━━━━━━━━━━━━━━━━━━━ 110s 529ms/step - accuracy: 0.8594 - loss: 0.4779 - val_accuracy: 0.9256 - val_loss: 0.1984
Epoch 3/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 0s 342ms/step - accuracy: 0.9481 - loss: 0.1360
Epoch 3: saving model to checkpoints/resnet34/resnet34_epoch-03.weights.h5
148/148 ━━━━━━━━━━━━━━━━━━━━ 58s 386ms/step - accuracy: 0.9516 - loss: 0.1297 - val_accuracy: 0.9485 - val_loss: 0.1414
Epoch 4/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 0s 343ms/step - accuracy: 0.9586 - loss: 0.1070
Epoch 4: saving model to checkpoints/resnet34/resnet34_epoch-04.weights.h5
148/148 ━━━━━━━━━━━━━━━━━━━━ 54s 366ms/step - accuracy: 0.9633 - loss: 0.0980 - val_accuracy: 0.9637 - val_loss: 0.1111
Epoch 5/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 0s 344ms/step - accuracy: 0.9676 - loss: 0.089

## Training History

The `fit` method returns a history object which we can use to plot training and validation accuracies and losses through epochs. The dataset is quite small, and the validation set is especially so, which is why validation metrics are a little jumpy.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_history(history):
    val_loss = history.history["val_loss"]
    best_epoch = int(np.argmin(val_loss))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(history.history["accuracy"], label="Train")
    ax1.plot(history.history["val_accuracy"], label="Validation")
    ax1.axvline(best_epoch, color="red", linestyle="--", alpha=0.5, label=f"Best epoch ({best_epoch + 1})")
    ax1.set_title("Accuracy")
    ax1.set_xlabel("Epoch")
    ax1.legend()

    ax2.plot(history.history["loss"], label="Train")
    ax2.plot(history.history["val_loss"], label="Validation")
    ax2.axvline(best_epoch, color="red", linestyle="--", alpha=0.5, label=f"Best epoch ({best_epoch + 1})")
    ax2.set_title("Loss")
    ax2.set_xlabel("Epoch")
    ax2.legend()

    plt.tight_layout()
    plt.show()

plot_history(history)

## Evaluation

With the class imbalance, accuracy can be slightly misleading. The following section evaluates more appropriate classification metrics against the test set (confusion matrix, auc-roc curve, precision-recall curve, f1). TODO.